In [1]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

In [2]:
model_path = "/mnt/storage_C1/igorzwirtes/poster_ic/qwen2.5coder"
adapter_path = "./qwen-spider-finetuned/r64q4a128_2"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [5]:
dataset = load_dataset("spider")
test_data = dataset["validation"]

In [6]:
SPIDER_TABLES_JSON = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/tables.json"

with open(SPIDER_TABLES_JSON) as f:
    tables_data = json.load(f)

# (mesma função do train.ipynb)
def format_schema_with_fk(db):
    col_names = db["column_names_original"]
    lines = []
    for i, table in enumerate(db["table_names_original"]):
        cols = [col[1] for col in col_names if col[0] == i]
        lines.append(f"  {table}({', '.join(cols)})")
    if db.get("foreign_keys"):
        lines.append("  Foreign keys:")
        for fk in db["foreign_keys"]:
            c1 = col_names[fk[0]]
            c2 = col_names[fk[1]]
            t1 = db["table_names_original"][c1[0]]
            t2 = db["table_names_original"][c2[0]]
            lines.append(f"    {t1}.{c1[1]} → {t2}.{c2[1]}")
    return "\n".join(lines)

schema_index = {db["db_id"]: format_schema_with_fk(db) for db in tables_data}

def build_prompt(example):
    schema = schema_index.get(example["db_id"], "")
    messages = [
        {"role": "system", "content": (
            "You are an expert SQL assistant. Given a natural language question and a database schema, "
            "generate the correct SQL query. "
            "Output ONLY the SQL query, with no explanation, no markdown, no comments."
        )},
        {"role": "user", "content": (
            f"Database: {example['db_id']}\n"
            f"Schema:\n{schema}\n\n"
            f"Question: {example['question']}"
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
predictions = []

for example in test_data:
    prompt = build_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=900).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=[
                tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids("<|im_end|>"),
            ],
        )

    # Decodifica só os tokens novos
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Pega só até o primeiro \n em branco ou marcador de explicação
    pred_sql = pred_sql.split("\n\n")[0].strip()

    # Remove blocos de markdown se o modelo ainda usar
    if "```sql" in pred_sql:
        pred_sql = pred_sql.split("```sql")[-1].split("```")[0].strip()
    elif "```" in pred_sql:
        pred_sql = pred_sql.split("```")[1].split("```")[0].strip()

    predictions.append({
        "db_id": example["db_id"],
        "question": example["question"],
        "gold": example["query"],
        "predicted": pred_sql,
    })
    print(f"Q: {example['question']}\nGold: {example['query']}\nPred: {pred_sql}\n---")

with open("predictions_r64q4a128_2.json", "w") as f:
    json.dump(predictions, f, indent=2)

print(f"\nPredições salvas: {len(predictions)} exemplos")

Q: How many singers do we have?
Gold: SELECT count(*) FROM singer
Pred: SELECT count(*) FROM singer
---
Q: What is the total number of singers?
Gold: SELECT count(*) FROM singer
Pred: SELECT count(*) FROM singer
---
Q: Show name, country, age for all singers ordered by age from the oldest to the youngest.
Gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Pred: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
---
Q: What are the names, countries, and ages for every singer in descending order of age?
Gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Pred: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
---
Q: What is the average, minimum, and maximum age of all singers from France?
Gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
Pred: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  "France"
---
Q: What is the average, minimum, and maximum age for all French singers?
Gold: SEL